In [1]:
import os
import json
import re
import numpy as np
from collections import defaultdict

BASE = "/kaggle/input/datasets/mnhlfuch/vqa-v2/VQA v2/VQA v2"
OUT_DIR = "/kaggle/working/preprocessed"

os.makedirs(OUT_DIR + "/Questions", exist_ok=True)
os.makedirs(OUT_DIR + "/Annotations", exist_ok=True)

TOP_K = 1000
regex = re.compile(r'\W+')

In [2]:
TRAIN_Q = BASE + "/Questions/v2_OpenEnded_mscoco_train2014_questions.json"
VAL_Q   = BASE + "/Questions/v2_OpenEnded_mscoco_val2014_questions.json"
TEST_Q  = BASE + "/Questions/v2_OpenEnded_mscoco_test2015_questions.json"

TRAIN_A = BASE + "/Annotations/v2_mscoco_train2014_annotations.json"
VAL_A   = BASE + "/Annotations/v2_mscoco_val2014_annotations.json"

## Question Vocab

In [9]:
import json

with open(TRAIN_Q, 'r') as f:
    data = json.load(f)

print(len(data['questions']))

print(data['questions'][0])

443757
{'image_id': 458752, 'question': 'What is this photo taken looking through?', 'question_id': 458752000}


In [3]:
def make_q_vocab():
    print("Building Question Vocabulary...")

    word_count = defaultdict(int)

    for path in [TRAIN_Q, VAL_Q]:
        with open(path, 'r') as f:
            data = json.load(f)

        for entry in data['questions']:
            question = entry['question'].lower()
            words = regex.sub(' ', question).split()

            for word in words:
                word_count[word] += 1

    save_path = OUT_DIR + "/Questions/question_vocab.txt"

    with open(save_path, 'w') as f:
        for word, count in word_count.items():
            f.write(f"{word} {count}\n")

    print("Saved at:", save_path)

## Answer Vocab

In [10]:
import json

with open(TRAIN_A, 'r') as f:
    data = json.load(f)

print(len(data['annotations']))

print(data['annotations'][0])

443757
{'question_type': 'what is this', 'multiple_choice_answer': 'net', 'answers': [{'answer': 'net', 'answer_confidence': 'maybe', 'answer_id': 1}, {'answer': 'net', 'answer_confidence': 'yes', 'answer_id': 2}, {'answer': 'net', 'answer_confidence': 'yes', 'answer_id': 3}, {'answer': 'netting', 'answer_confidence': 'yes', 'answer_id': 4}, {'answer': 'net', 'answer_confidence': 'yes', 'answer_id': 5}, {'answer': 'net', 'answer_confidence': 'yes', 'answer_id': 6}, {'answer': 'mesh', 'answer_confidence': 'maybe', 'answer_id': 7}, {'answer': 'net', 'answer_confidence': 'yes', 'answer_id': 8}, {'answer': 'net', 'answer_confidence': 'yes', 'answer_id': 9}, {'answer': 'net', 'answer_confidence': 'yes', 'answer_id': 10}], 'image_id': 458752, 'answer_type': 'other', 'question_id': 458752000}


In [5]:
def clean_answer(ans):
    ans = ans.lower()
    ans = re.sub(r'[^\w\s]', '', ans)
    return ans.strip()

In [6]:
def make_a_vocab(top_answer):
    print("Building Answer Vocabulary...")

    answers = defaultdict(int)

    for path in [TRAIN_A, VAL_A]:
        with open(path, 'r') as f:
            data = json.load(f)

        for entry in data['annotations']:
            ans = clean_answer(entry['multiple_choice_answer'])

            if ans == "":
                continue

            answers[ans] += 1

    sorted_ans = sorted(answers.items(), key=lambda x: x[1], reverse=True)[:top_answer]

    save_path = OUT_DIR + "/Annotations/answer_vocab.txt"

    with open(save_path, 'w') as f:
        for ans, count in sorted_ans:
            f.write(f"{ans} {count}\n")

    print("Saved at:", save_path)

## Running Vocab Creation

In [8]:
make_q_vocab()
make_a_vocab(TOP_K)

Building Question Vocabulary...
Saved at: /kaggle/working/preprocessed/Questions/question_vocab.txt
Building Answer Vocabulary...
Saved at: /kaggle/working/preprocessed/Annotations/answer_vocab.txt


## Preprocessing

### Load Top Answers

In [12]:
def load_top_answers():
    path = OUT_DIR + "/Annotations/answer_vocab.txt"
    top_answers = []

    with open(path, 'r') as f:
        for line in f:
            ans, _ = line.rsplit(" ", 1)
            top_answers.append(ans)

    return top_answers

### Match Top Answer

In [13]:
def match_top_ans(ans, top_answers):
    if ans not in top_answers:
        match_top_ans.unk += 1
        return "<unk>"
    return ans

match_top_ans.unk = 0
#unk is like a global variable for the function match_top_ans

### Preprocessing function

In [14]:
def preprocessing(question_path, annotation_path=None):

    print("Processing:", question_path)

    with open(question_path, 'r') as f:
        q_data = json.load(f)

    questions = q_data['questions']

    annotations = {}
    if annotation_path:
        with open(annotation_path, 'r') as f:
            a_data = json.load(f)

        annotations = {a['question_id']: a for a in a_data['annotations']}

    top_answers = load_top_answers()

    processed = []

    for q in questions:
        qid = q['question_id']
        image_id = q['image_id']
        question = q['question']

        if annotation_path:
            ans = clean_answer(annotations[qid]['multiple_choice_answer']).lower()
            ans = match_top_ans(ans, top_answers)
        else:
            ans = None

        processed.append({
            "question": question,
            "image_id": image_id,
            "answer": ans
        })

    return processed

In [15]:
train_data = preprocessing(TRAIN_Q, TRAIN_A)
val_data   = preprocessing(VAL_Q, VAL_A)
test_data  = preprocessing(TEST_Q, None)

Processing: /kaggle/input/datasets/mnhlfuch/vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_train2014_questions.json
Processing: /kaggle/input/datasets/mnhlfuch/vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_val2014_questions.json
Processing: /kaggle/input/datasets/mnhlfuch/vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_test2015_questions.json


In [16]:
np.save(OUT_DIR + "/train.npy", np.array(train_data, dtype=object))
np.save(OUT_DIR + "/val.npy", np.array(val_data, dtype=object))
np.save(OUT_DIR + "/test.npy", np.array(test_data, dtype=object))

print("Saved all splits!")
print("Unknown answers:", match_top_ans.unk)

Saved all splits!
Unknown answers: 82941


### Total Questions = 443757
### Unknown answers = 82941 
### Unknown answers ratio = 82941 / 443757 ≈ 18.7%